In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Labs"
)

xpt_file = DATA_DIR / "GLU_J.xpt"
csv_file = DATA_DIR / "GLU_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#

source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 GLU_J XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXGLU",
    "LBDGLUSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "GLU_J column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 3036:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 4:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 3,036 rows x 4 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# WHOLE-NUMBER FIELDS
# ------------------------------------------------------------
#
# SEQN
#     respondent identifier
#
# LBXGLU
#     fasting glucose, mg/dL
#
#
# GENUINE DECIMAL FIELDS
# ------------------------------------------------------------
#
# WTSAF2YR
#     fasting subsample survey weight
#
# LBDGLUSI
#     fasting glucose, mmol/L
#
#
# IMPORTANT:
#
# WTSAF2YR and LBDGLUSI must NOT be rounded.
#
# They must NOT be converted to Int64.
#
# They must NOT use forced float formatting.
#

INTEGER_COLUMNS = [
    "SEQN",
    "LBXGLU"
]


DECIMAL_COLUMNS = [
    "WTSAF2YR",
    "LBDGLUSI"
]


if set(
    INTEGER_COLUMNS
    +
    DECIMAL_COLUMNS
) != set(EXPECTED_COLUMNS):

    raise ValueError(
        "Field type plan does not cover "
        "all GLU_J variables."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# ============================================================
# 5. VERIFY ALL FIELDS LOADED NUMERICALLY
# ============================================================

numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 4 GLU_J variables "
        "to load numerically.\n\n"
        f"Actual numeric columns:\n{numeric_cols}"
    )


print(
    "PASS: All 4 GLU_J variables loaded numerically."
)


# ============================================================
# 6. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Never globally replace this value without validating
# exactly where it occurs.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[EXPECTED_COLUMNS]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before repair:",
    true_zero_total
)

print(
    "Tiny XPORT artifacts:",
    tiny_total
)

print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 7. VALIDATE EXACT TINY-VALUE PATTERN
# ============================================================

#
# Attached GLU_J:
#
#     WTSAF2YR = 325 tiny values
#
# CDC:
#
#     WTSAF2YR = 0
#     Count = 325
#
# Therefore these are validated zero restorations.
#

EXPECTED_TINY_COUNTS = {
    "WTSAF2YR": 325
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: GLU_J tiny-value pattern differs "
        "from validated attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 325:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zero values existed "
        "before correction."
    )


print(
    "PASS: Exact 325-value WTSAF2YR "
    "tiny-zero pattern validated."
)


# ============================================================
# 8. RESTORE ONLY VALIDATED WTSAF2YR ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

weight_tiny_mask = (
    df["WTSAF2YR"]
    .eq(TINY_VALUE)
)


correction_count = int(
    weight_tiny_mask.sum()
)


if correction_count != 325:

    raise ValueError(
        "Unexpected WTSAF2YR tiny-value count."
    )


df.loc[
    weight_tiny_mask,
    "WTSAF2YR"
] = 0


print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)

print(
    "WTSAF2YR restored to zero:",
    correction_count
)


if int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
) != 325:

    raise ValueError(
        "WTSAF2YR zero restoration failed."
    )


print(
    "PASS: 325 validated WTSAF2YR zeros restored."
)


# ============================================================
# 9. CREATE EXPECTED CORRECTED SOURCE
# ============================================================

#
# The ONLY permitted source modification:
#
#     WTSAF2YR tiny artifact -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WTSAF2YR"]
    .eq(TINY_VALUE),
    "WTSAF2YR"
] = 0


# ============================================================
# 10. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny XPORT artifacts remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 11. FRACTIONAL VALUE DISCOVERY
# ============================================================

#
# Run BEFORE converting any variable to Int64.
#
# np.round() below is used ONLY for comparison.
#
# It DOES NOT modify df.
#

print(
    "\n--- FRACTIONAL VALUE DISCOVERY ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:

        fractional_count = 0

    else:

        fractional_mask = ~np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        fractional_count = int(
            fractional_mask.sum()
        )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


EXPECTED_FRACTIONAL_COUNTS = {
    "SEQN": 0,
    "WTSAF2YR": 2711,
    "LBXGLU": 0,
    "LBDGLUSI": 2769
}


if fractional_counts != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Unexpected fractional-value pattern.\n\n"
        f"Expected:\n{EXPECTED_FRACTIONAL_COUNTS}\n\n"
        f"Actual:\n{fractional_counts}"
    )


print(
    "PASS: Whole-number and decimal fields "
    "identified correctly."
)


# ============================================================
# 12. PROVE INTEGER FIELDS ARE SAFE
# ============================================================

print(
    "\n--- INTEGER FIELD VALIDATION ---"
)


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        (~whole_mask)
        .sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional values"
    )


    if fractional_count != 0:

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: SEQN and LBXGLU are proven "
    "whole-number fields."
)


# ============================================================
# 13. VALIDATE SEQN
# ============================================================

print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93708:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93708-102956 validated."
)


# ============================================================
# 14. VALIDATE WTSAF2YR
# ============================================================

positive_weight_mask = (
    df["WTSAF2YR"]
    .gt(0)
)


positive_weight_count = int(
    positive_weight_mask.sum()
)


weight_zero_count = int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
)


weight_missing_count = int(
    df["WTSAF2YR"]
    .isna()
    .sum()
)


weight_min_positive = float(
    df.loc[
        positive_weight_mask,
        "WTSAF2YR"
    ]
    .min()
)


weight_max_positive = float(
    df.loc[
        positive_weight_mask,
        "WTSAF2YR"
    ]
    .max()
)


print(
    "\n--- WTSAF2YR CHECK ---"
)

print(
    "Positive weights:",
    positive_weight_count
)

print(
    "Zero weights:",
    weight_zero_count
)

print(
    "Missing weights:",
    weight_missing_count
)

print(
    "Minimum positive:",
    repr(
        weight_min_positive
    )
)

print(
    "Maximum positive:",
    repr(
        weight_max_positive
    )
)


if positive_weight_count != 2711:

    raise ValueError(
        "Unexpected WTSAF2YR positive count."
    )


if weight_zero_count != 325:

    raise ValueError(
        "Unexpected WTSAF2YR zero count."
    )


if weight_missing_count != 0:

    raise ValueError(
        "Unexpected WTSAF2YR missing values."
    )


if not np.isclose(
    weight_min_positive,
    9133.518063,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected minimum positive WTSAF2YR."
    )


if not np.isclose(
    weight_max_positive,
    944153.249752,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected maximum positive WTSAF2YR."
    )


print(
    "PASS: WTSAF2YR distribution validated."
)


# ============================================================
# 15. CDC CHECK - LBXGLU
# ============================================================

#
# CDC:
#
# Fasting glucose mg/dL
#
# Range:
#
#     47 - 451
#
# Nonmissing:
#
#     2,891
#
# Missing:
#
#     145
#

glu_nonmissing = int(
    df["LBXGLU"]
    .notna()
    .sum()
)


glu_missing = int(
    df["LBXGLU"]
    .isna()
    .sum()
)


glu_min = float(
    df["LBXGLU"]
    .min()
)


glu_max = float(
    df["LBXGLU"]
    .max()
)


print(
    "\n--- CDC CHECK: LBXGLU ---"
)

print(
    "Nonmissing:",
    glu_nonmissing
)

print(
    "Missing:",
    glu_missing
)

print(
    "Minimum:",
    glu_min
)

print(
    "Maximum:",
    glu_max
)


if glu_nonmissing != 2891:

    raise ValueError(
        "LBXGLU nonmissing count differs from CDC."
    )


if glu_missing != 145:

    raise ValueError(
        "LBXGLU missing count differs from CDC."
    )


if glu_min != 47:

    raise ValueError(
        "LBXGLU minimum differs from CDC."
    )


if glu_max != 451:

    raise ValueError(
        "LBXGLU maximum differs from CDC."
    )


print(
    "PASS: LBXGLU matches CDC."
)


# ============================================================
# 16. CDC CHECK - LBDGLUSI
# ============================================================

#
# CDC:
#
# Fasting glucose mmol/L
#
# Range:
#
#     2.61 - 25
#
# Nonmissing:
#
#     2,891
#
# Missing:
#
#     145
#

si_nonmissing = int(
    df["LBDGLUSI"]
    .notna()
    .sum()
)


si_missing = int(
    df["LBDGLUSI"]
    .isna()
    .sum()
)


si_min = float(
    df["LBDGLUSI"]
    .min()
)


si_max = float(
    df["LBDGLUSI"]
    .max()
)


print(
    "\n--- CDC CHECK: LBDGLUSI ---"
)

print(
    "Nonmissing:",
    si_nonmissing
)

print(
    "Missing:",
    si_missing
)

print(
    "Minimum:",
    si_min
)

print(
    "Maximum:",
    si_max
)


if si_nonmissing != 2891:

    raise ValueError(
        "LBDGLUSI nonmissing count differs from CDC."
    )


if si_missing != 145:

    raise ValueError(
        "LBDGLUSI missing count differs from CDC."
    )


if not np.isclose(
    si_min,
    2.61,
    rtol=0,
    atol=0
):

    raise ValueError(
        "LBDGLUSI minimum differs from CDC."
    )


if not np.isclose(
    si_max,
    25.0,
    rtol=0,
    atol=0
):

    raise ValueError(
        "LBDGLUSI maximum differs from CDC."
    )


print(
    "PASS: LBDGLUSI matches CDC."
)


# ============================================================
# 17. MISSING-POSITION ALIGNMENT
# ============================================================

#
# LBXGLU and LBDGLUSI represent the same measurement
# in different units.
#

missing_position_mismatch = int(
    np.sum(
        df["LBXGLU"]
        .isna()
        .to_numpy()
        !=
        df["LBDGLUSI"]
        .isna()
        .to_numpy()
    )
)


print(
    "\n--- LBXGLU / LBDGLUSI MISSING ALIGNMENT ---"
)

print(
    "Missing-position mismatches:",
    missing_position_mismatch
)


if missing_position_mismatch != 0:

    raise ValueError(
        "LBXGLU and LBDGLUSI missing positions differ."
    )


print(
    "PASS: Missing positions match exactly."
)


# ============================================================
# 18. GLUCOSE CONVERSION CONSISTENCY
# ============================================================

#
# CDC documentation states:
#
#     LBDGLUSI = LBXGLU * 0.05551
#
# However, DO NOT use exact arithmetic equality here.
#
# Why?
#
# LBXGLU itself is published as whole mg/dL while the
# underlying laboratory measurement can have greater
# precision.
#
# Therefore recomputing SI from the already-published
# rounded mg/dL value can legitimately differ from
# the released SI field.
#
# The CDC documentation is used as a conceptual
# consistency check only.
#
# We DO NOT recalculate LBDGLUSI.
#

conversion_df = (
    df[
        [
            "LBXGLU",
            "LBDGLUSI"
        ]
    ]
    .dropna()
)


derived_from_published_mgdl = (
    conversion_df["LBXGLU"]
    *
    0.05551
)


conversion_difference = (
    conversion_df["LBDGLUSI"]
    -
    derived_from_published_mgdl
).abs()


max_conversion_difference = float(
    conversion_difference.max()
)


print(
    "\n--- GLUCOSE CONVERSION DIAGNOSTIC ---"
)

print(
    "Records checked:",
    len(conversion_df)
)

print(
    "Maximum difference from published "
    "LBXGLU * 0.05551:",
    max_conversion_difference
)


#
# This is diagnostic only.
#
# DO NOT fail based on exact conversion because the
# published mg/dL field has already been rounded.
#

print(
    "INFO: Conversion diagnostic only; "
    "LBDGLUSI is not recalculated."
)


# ============================================================
# 19. DECIMAL FIELD PRE-CONVERSION FIDELITY
# ============================================================

print(
    "\n--- DECIMAL FIELD PRE-CONVERSION FIDELITY ---"
)


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Decimal fields changed unexpectedly:\n"
        + str(decimal_failures)
    )


print(
    "PASS: WTSAF2YR and LBDGLUSI unchanged "
    "apart from validated WTSAF2YR zeros."
)


# ============================================================
# 20. CONVERT ONLY PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# Convert:
#
#     SEQN
#     LBXGLU
#
# to nullable Int64.
#
# DO NOT TOUCH:
#
#     WTSAF2YR
#     LBDGLUSI
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        raise ValueError(
            f"{col} cannot safely be converted "
            "to Int64."
        )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN and LBXGLU converted "
    "to nullable Int64."
)

print(
    "PASS: WTSAF2YR and LBDGLUSI retained "
    "as genuine decimal fields."
)


# ============================================================
# 21. COMPLETE PRE-EXPORT VALUE FIDELITY
# ============================================================

print(
    "\n--- COMPLETE PRE-EXPORT VALUE FIDELITY ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All GLU_J values match exact "
    "expected corrected source values."
)


# ============================================================
# 22. MISSING-VALUE FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    original_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            original_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position differences before export:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved."
)


# ============================================================
# 23. FINAL ZERO PATTERN BEFORE EXPORT
# ============================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if count > 0:

        final_zero_counts[col] = count


EXPECTED_FINAL_ZERO_COUNTS = {
    "WTSAF2YR": 325
}


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if (
    final_zero_counts
    !=
    EXPECTED_FINAL_ZERO_COUNTS
):

    raise ValueError(
        "Unexpected final zero pattern.\n\n"
        f"Expected:\n{EXPECTED_FINAL_ZERO_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern exactly correct."
)


# ============================================================
# 24. EXPORT CSV
# ============================================================

#
# CRITICAL FIDELITY POLICY
# ------------------------------------------------------------
#
# DO NOT:
#
#     df.round(...)
#
# DO NOT:
#
#     float_format=
#
# DO NOT:
#
#     convert WTSAF2YR to integer
#
# DO NOT:
#
#     convert LBDGLUSI to integer
#
# DO NOT:
#
#     recalculate LBDGLUSI
#
#
# Genuine decimal values must remain AS IS.
#
# This is the same principle used for Dietary,
# Laboratory, Examination, survey weight, nutrient,
# and exposure variables.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 25. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

#
# IMPORTANT:
#
# Exact floating-point fidelity is checked with:
#
#     float_precision="round_trip"
#
# NOT by reparsing raw CSV strings with pd.to_numeric().
#

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 26. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV fidelity differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 4 variables round-trip exactly."
)


# ============================================================
# 27. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- CSV MISSING-VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing positions changed during CSV export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 28. READ FINAL CSV AS RAW TEXT
# ============================================================

#
# Raw text is used ONLY for:
#
#     .0 checks
#     literal zero checks
#     scientific notation checks
#
# It is NOT used for exact float fidelity.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    3036,
    4
):

    raise ValueError(
        "Raw CSV dimensions changed."
    )


if (
    raw_csv_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "Raw CSV column order changed."
    )


# ============================================================
# 29. INTEGER-FIELD .0 CHECK
# ============================================================

#
# ONLY:
#
#     SEQN
#     LBXGLU
#
# Do NOT check WTSAF2YR or LBDGLUSI.
#

print(
    "\n--- INTEGER-FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: SEQN and LBXGLU contain no "
    "unnecessary '.0'."
)


# ============================================================
# 30. DECIMAL FIELD RAW-TEXT CHECK
# ============================================================

#
# Do NOT remove ".0" from decimal fields.
#
# Example:
#
# LBDGLUSI may legitimately contain a value that
# numerically equals:
#
#     25.0
#
# The VARIABLE remains a decimal measurement.
#

print(
    "\n--- DECIMAL FIELD CSV CHECK ---"
)


for col in DECIMAL_COLUMNS:

    nonblank_count = int(
        raw_csv_df[col]
        .ne("")
        .sum()
    )


    dot_zero_count = int(
        (
            raw_csv_df[col]
            .ne("")
            &
            raw_csv_df[col]
            .str.endswith(".0")
        )
        .sum()
    )


    print(
        f"{col} populated:",
        nonblank_count
    )

    print(
        f"{col} tokens ending '.0' "
        "(allowed):",
        dot_zero_count
    )


print(
    "PASS: Decimal variables retained "
    "without forced formatting."
)


# ============================================================
# 31. WTSAF2YR ROUND-TRIP PRECISION
# ============================================================

expected_weight_values = (
    expected_df["WTSAF2YR"]
    .to_numpy(
        dtype=float
    )
)


final_weight_values = (
    roundtrip_df["WTSAF2YR"]
    .to_numpy(
        dtype=float
    )
)


weight_exact = np.allclose(
    expected_weight_values,
    final_weight_values,
    rtol=0,
    atol=0,
    equal_nan=True
)


print(
    "\n--- WTSAF2YR EXACT FIDELITY ---"
)

print(
    "Exact round-trip:",
    weight_exact
)


if not weight_exact:

    raise ValueError(
        "WTSAF2YR changed during CSV export."
    )


print(
    "PASS: WTSAF2YR preserved exactly."
)


# ============================================================
# 32. LBDGLUSI ROUND-TRIP PRECISION
# ============================================================

expected_si_values = (
    expected_df["LBDGLUSI"]
    .to_numpy(
        dtype=float
    )
)


final_si_values = (
    roundtrip_df["LBDGLUSI"]
    .to_numpy(
        dtype=float
    )
)


si_exact = np.allclose(
    expected_si_values,
    final_si_values,
    rtol=0,
    atol=0,
    equal_nan=True
)


print(
    "\n--- LBDGLUSI EXACT FIDELITY ---"
)

print(
    "Exact round-trip:",
    si_exact
)


if not si_exact:

    raise ValueError(
        "LBDGLUSI changed during CSV export."
    )


print(
    "PASS: LBDGLUSI decimal values "
    "preserved exactly."
)


# ============================================================
# 33. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY-VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 34. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    count = int(
        values.eq(0)
        .sum()
    )


    if count > 0:

        csv_zero_counts[col] = count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if (
    csv_zero_counts
    !=
    EXPECTED_FINAL_ZERO_COUNTS
):

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_FINAL_ZERO_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "325 WTSAF2YR zeros."
)


# ============================================================
# 35. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_errors = {}


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    count = int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


    if count > 0:

        scientific_errors[col] = count


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    scientific_errors
)


if scientific_errors:

    raise ValueError(
        "Unexpected scientific notation found:\n"
        + str(scientific_errors)
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 36. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93708:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN 93708-102956 valid "
    "and contains no '.0'."
)


# ============================================================
# 37. FINAL DECIMAL-FIELD EXACT FIDELITY
# ============================================================

#
# IMPORTANT:
#
# Use roundtrip_df.
#
# Do NOT use pd.to_numeric(raw text) for exact binary
# float equality.
#

print(
    "\n--- FINAL DECIMAL-FIELD EXACT FIDELITY ---"
)


decimal_csv_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    print(
        f"{col}:",
        "PASS" if same else "FAIL"
    )


    if not same:

        decimal_csv_failures.append(
            col
        )


if decimal_csv_failures:

    raise ValueError(
        "Decimal fidelity failures:\n"
        + str(decimal_csv_failures)
    )


print(
    "PASS: Every genuine decimal variable "
    "round-trips exactly."
)


# ============================================================
# 38. FINAL ALL-COLUMN SOURCE -> CSV FIDELITY
# ============================================================

print(
    "\n--- FINAL ALL-COLUMN SOURCE -> CSV FIDELITY ---"
)


final_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        final_fidelity_failures.append(
            col
        )


if final_fidelity_failures:

    raise ValueError(
        "Final source -> CSV fidelity failures:\n"
        + str(final_fidelity_failures)
    )


print(
    "PASS: Final CSV preserves all expected "
    "GLU_J numeric values exactly."
)


# ============================================================
# 39. ROUND-TRIP MAXIMUM DIFFERENCE DIAGNOSTIC
# ============================================================

print(
    "\n--- ROUND-TRIP MAXIMUM ABSOLUTE DIFFERENCES ---"
)


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    valid_mask = (
        ~np.isnan(expected_values)
        &
        ~np.isnan(final_values)
    )


    if valid_mask.any():

        max_difference = float(
            np.max(
                np.abs(
                    expected_values[valid_mask]
                    -
                    final_values[valid_mask]
                )
            )
        )

    else:

        max_difference = 0.0


    print(
        f"{col}: {max_difference}"
    )


# ============================================================
# 40. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "GLU_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Laboratory - "
    "Plasma Fasting Glucose"
)

print(
    "Eligible population: "
    "Morning examined participants age 12+"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "WTSAF2YR zero restorations:",
    correction_count
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Positive fasting weights:",
    positive_weight_count
)

print(
    "Zero fasting weights:",
    weight_zero_count
)

print(
    "WTSAF2YR positive range:",
    f"{weight_min_positive} - "
    f"{weight_max_positive}"
)

print(
    "LBXGLU nonmissing:",
    glu_nonmissing
)

print(
    "LBXGLU range:",
    f"{glu_min}-{glu_max}"
)

print(
    "LBDGLUSI nonmissing:",
    si_nonmissing
)

print(
    "LBDGLUSI range:",
    f"{si_min}-{si_max}"
)

print(
    "WTSAF2YR fractional observations:",
    fractional_counts["WTSAF2YR"]
)

print(
    "LBDGLUSI fractional observations:",
    fractional_counts["LBDGLUSI"]
)

print(
    "Exact decimal round-trip failures: "
    "0 expected"
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0' in integer fields: "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 GLU_J XPT -> CSV
Rows: 3,036
Columns: 4
PASS: Structure = 3,036 rows x 4 columns.

--- FIELD TYPE PLAN ---
Whole-number fields: ['SEQN', 'LBXGLU']
Genuine decimal fields: ['WTSAF2YR', 'LBDGLUSI']
PASS: All 4 GLU_J variables loaded numerically.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before repair: 0
Tiny XPORT artifacts: 325

Tiny values by variable:
WTSAF2YR    325
PASS: Exact 325-value WTSAF2YR tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
WTSAF2YR restored to zero: 325
PASS: 325 validated WTSAF2YR zeros restored.

Tiny XPORT artifacts remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE DISCOVERY ---
SEQN: 0 fractional observations
WTSAF2YR: 2,711 fractional observations
LBXGLU: 0 fractional observations
LBDGLUSI: 2,769 fractional observations
PASS: Whole-number and decimal fields identified correctly.

--- INTEGER FIELD VALIDATION ---
SEQN: 0 fractional values
LBXGLU: 0 fractional values
PASS: SEQN and LBXGLU a